In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
house_data = pd.read_csv("/Users/ashishkrsingh/Developers/Machine Learning/Projects/House-Price-Prediction/data/train.csv")

# Data Cleaning & Preprocessing

In [ ]:
X = house_data.drop(columns=["Price_INR_Lakhs"])
y = house_data["Price_INR_Lakhs"]

In [ ]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
print(f"Price_INR_Lakhs in X_train: {"Price_INR_Lakhs" in X_train.columns}")
print(f"Price_INR_Lakhs in X_test: {"Price_INR_Lakhs" in X_test.columns}")

In [ ]:
print("Overlapping rows:",
      len(X_train.index.intersection(X_test.index)))

In [ ]:
print("Training Target: ")
print(y_train.describe())
print("Test Target: ")
print(y_test.describe())

In [ ]:
X_train = X_train.drop(columns=["Property_ID"])
X_test = X_test.drop(columns=["Property_ID"])

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(X_train.dtypes)

In [ ]:
print("Numerical Features: ")
print(X_train.select_dtypes(include="number").columns.to_list())

print("Categorical Features: ")
print(X_train.select_dtypes(include="object").columns.to_list())

In [ ]:
missing = X_train.isnull().sum()
print(missing)

print("\nColumns with missing values:")
print(missing[missing > 0])

In [ ]:
categorical_features = X_train.select_dtypes(include="object").columns.to_list()

for columns in categorical_features:
    print(f"\n{columns}")
    print(X_train[columns].value_counts())

In [ ]:
categorical_features = X_train.select_dtypes(include="object").columns

for column in categorical_features:
    train_categories = set(X_train[column].unique())
    test_categories = set(X_test[column].unique())

    print(f"\n{column}")
    print("Categories only in test:",
          test_categories - train_categories)

In [ ]:
numerical_features = X_train.select_dtypes(include="number").columns.to_list()
categorical_features = X_train.select_dtypes(include="object").columns.to_list()

print("Numerical Features: ", numerical_features)
print("Number of Numerical Features: ", len(numerical_features))
print("Categorical Features: ", categorical_features)
print("Number of Categorical Features: ", len(categorical_features))

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[numerical_features])

In [ ]:
print(scaler.mean_)
print(scaler.scale_)

In [ ]:
X_train_numerical_scaled = scaler.transform(X_train[numerical_features])

In [ ]:
print(X_train_numerical_scaled.shape)

In [ ]:
print(X_train_numerical_scaled.mean(axis=0))
print(X_train_numerical_scaled.std(axis=0))

In [ ]:
X_test_numerical_scaled = scaler.transform(X_test[numerical_features])

In [ ]:
print(X_test_numerical_scaled.shape)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore")

In [ ]:
encoder.fit(X_train[categorical_features])

In [ ]:
print(encoder.categories_)

In [ ]:
X_train_categorical_encoded = encoder.transform(X_train[categorical_features])

In [ ]:
print(X_train_categorical_encoded.shape)

In [ ]:
X_test_categorical_encoded = encoder.transform(X_test[categorical_features])

In [ ]:
print(X_test_categorical_encoded.shape)

In [ ]:
encoded_features_names = encoder.get_feature_names_out(categorical_features)

print(encoded_features_names)
print(f"Number of Encoded Features: {len(encoded_features_names)}")

In [ ]:
print("Scaled numerical shape:", X_train_numerical_scaled.shape)
print("Encoded categorical shape:", X_train_categorical_encoded.shape)

print(
    "Total features:",
    X_train_numerical_scaled.shape[1]
    + X_train_categorical_encoded.shape[1]
)

In [ ]:
X_train_processed = np.hstack((X_train_numerical_scaled, X_train_categorical_encoded.toarray()))

In [ ]:
X_test_processed = np.hstack((X_test_numerical_scaled, X_test_categorical_encoded.toarray()))

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
print("NaN in X_train:", np.isnan(X_train_processed).sum())
print("NaN in X_test:", np.isnan(X_test_processed).sum())

print("Inf in X_train:", np.isinf(X_train_processed).sum())
print("Inf in X_test:", np.isinf(X_test_processed).sum())

In [ ]:
processed_feature_names = (
    numerical_features
    + list(encoded_features_names)
)

print("Number of feature names:", len(processed_feature_names))
print(processed_feature_names)

In [ ]:
print("Columns: ", X_train_processed.shape[1])
print("Features Name: ", len(processed_feature_names))

# Linear Regression

In [ ]:
X_train_bias = np.c_[np.ones(X_train_processed.shape[0]),X_train_processed]

In [ ]:
X_test_bias = np.c_[np.ones(X_test_processed.shape[0]),X_test_processed]

In [ ]:
print(f"X_train_bias.shape: {X_train_bias.shape}")
print(f"X_test_bias.shape: {X_test_bias.shape}")

In [ ]:
class LinearRegression():

    def __init__(self):
        self.theta = None

    def fit(self, X, y):
        # Calculate theta using the Normal Equation
        self.theta = np.linalg.inv(X.T @ X) @ X.T @ y

    def predict(self, X):
        return X @ self.theta

In [ ]:
model = LinearRegression()
model.fit(X_train_bias, y_train)

In [ ]:
print(model.theta)
print("Number of parameters:", len(model.theta))

In [ ]:
y_pred = model.predict(X_test_bias)

In [ ]:
print(y_pred.shape)
print(y_pred[:10])

In [ ]:
errors = y_test - y_pred

In [ ]:
print(errors[:10])
print("Mean Error: ", np.mean(errors))

In [ ]:
print("Number of columns:", X_train_bias.shape[1])
print("Matrix rank:", np.linalg.matrix_rank(X_train_bias))

In [ ]:
encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)

encoder.fit(X_train[categorical_features])

encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

print(encoded_feature_names)
print("Number of encoded features:", len(encoded_feature_names))

In [ ]:
X_train_categorical_encoded = encoder.transform(
    X_train[categorical_features]
)

print(X_train_categorical_encoded.shape)

In [ ]:
X_test_categorical_encoded = encoder.transform(
    X_test[categorical_features]
)

print(X_test_categorical_encoded.shape)

In [ ]:
X_train_processed = np.hstack([
    X_train_numerical_scaled,
    X_train_categorical_encoded.toarray()
])

print(X_train_processed.shape)

In [ ]:
X_test_processed = np.hstack([
    X_test_numerical_scaled,
    X_test_categorical_encoded.toarray()
])

print(X_test_processed.shape)

In [ ]:
X_train_bias = np.c_[np.ones((X_train_processed.shape[0])), X_train_processed]

In [ ]:
X_test_bias = np.c_[np.ones(X_test_processed.shape[0]),X_test_processed]

In [ ]:
print("X_train_with_bias:", X_train_bias.shape)
print("X_test_with_bias:", X_test_bias.shape)

print("Number of columns:",
      X_train_bias.shape[1])

print("Matrix rank:",
      np.linalg.matrix_rank(X_train_bias))

In [ ]:
model = LinearRegression()

model.fit(
    X_train_bias,
    y_train
)

In [ ]:
print("Number of parameters:", len(model.theta))
print(model.theta)

In [ ]:
y_pred = model.predict(X_test_bias)

In [ ]:
errors = y_test - y_pred
print(errors[:10])
print("Mean Error: ", np.mean(errors))

In [ ]:
mae = np.mean(np.abs(errors))
print("Mean Absolute Error (MAE):", mae)

In [ ]:
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)
print("Root Mean Square Error (RMSE):", rmse)

In [ ]:
ssr = np.sum(errors ** 2)
tss = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - (ssr / tss)
print("R-squared:", r2)

In [ ]:
abs_errors = np.abs(errors.to_numpy())

largest_error_indices = np.argsort(abs_errors)[-10:][::-1]

print(abs_errors[largest_error_indices])

In [ ]:
largest_error_rows = X_test.iloc[largest_error_indices].copy()

largest_error_rows["Actual_Price"] = y_test.iloc[
    largest_error_indices
].values

largest_error_rows["Predicted_Price"] = y_pred[
    largest_error_indices
]

largest_error_rows["Absolute_Error"] = abs_errors[
    largest_error_indices
]

print(largest_error_rows)

In [ ]:
worst_predictions = house_data.loc[largest_error_rows.index].copy()

worst_predictions["Predicted_Price"] = y_pred[largest_error_indices]
worst_predictions["Absolute_Error"] = abs_errors[largest_error_indices]

print(worst_predictions)

In [ ]:
for name, coefficient in zip(
    ["Intercept"] + processed_feature_names,
    model.theta
):
    print(f"{name}: {coefficient}")

In [ ]:
condition_number = np.linalg.cond(
    X_train_bias.T @ X_train_bias
)

print("Condition number:", condition_number)

In [ ]:
carpet_index = numerical_features.index("Carpet_Area_SqFt")

X_train_no_carpet = np.delete(
    X_train_processed,
    carpet_index,
    axis=1
)

X_test_no_carpet = np.delete(
    X_test_processed,
    carpet_index,
    axis=1
)

print(X_train_no_carpet.shape)
print(X_test_no_carpet.shape)

In [ ]:
model_no_carpet = LinearRegression()

model_no_carpet.fit(
    X_train_no_carpet,
    y_train.to_numpy()
)

In [ ]:
y_pred_no_carpet = model_no_carpet.predict(
    X_test_no_carpet
)

In [ ]:
errors_no_carpet = y_test.to_numpy() - y_pred_no_carpet

mae_no_carpet = np.mean(np.abs(errors_no_carpet))

mse_no_carpet = np.mean(errors_no_carpet ** 2)
rmse_no_carpet = np.sqrt(mse_no_carpet)

ss_res_no_carpet = np.sum(errors_no_carpet ** 2)
ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2_no_carpet = 1 - (ss_res_no_carpet / ss_tot)

print("MAE:", mae_no_carpet)
print("RMSE:", rmse_no_carpet)
print("R²:", r2_no_carpet)

In [ ]:
print("Condition number of X:",
      np.linalg.cond(X_train_bias))

In [ ]:
y_train_log = np.log1p(y_train.to_numpy())

print(y_train_log[:10])

In [ ]:
model_log = LinearRegression()

model_log.fit(
    X_train_bias,
    y_train_log
)

In [ ]:
print("Number of parameters:", len(model_log.theta))

In [ ]:
y_pred_log = model_log.predict(X_test_bias)

print(y_pred_log[:10])

In [ ]:
y_pred_log_original = np.expm1(y_pred_log)

In [ ]:
errors_log_model = y_test.to_numpy() - y_pred_log_original

mae_log_model = np.mean(np.abs(errors_log_model))

mse_log_model = np.mean(errors_log_model ** 2)
rmse_log_model = np.sqrt(mse_log_model)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum((y_test_array - y_pred_log_original) ** 2)
ss_tot = np.sum((y_test_array - y_mean) ** 2)

r2_log_model = 1 - (ss_res / ss_tot)

print("MAE :", mae_log_model)
print("RMSE:", rmse_log_model)
print("R²  :", r2_log_model)

In [ ]:
abs_errors_log_model = np.abs(
    y_test.to_numpy() - y_pred_log_original
)

largest_error_indices_log = np.argsort(
    abs_errors_log_model
)[-10:][::-1]

print(abs_errors_log_model[largest_error_indices_log])

In [ ]:
worst_log_predictions = X_test.iloc[
    largest_error_indices_log
].copy()

worst_log_predictions["Actual_Price"] = y_test.iloc[
    largest_error_indices_log
].values

worst_log_predictions["Predicted_Price"] = y_pred_log_original[
    largest_error_indices_log
]

worst_log_predictions["Absolute_Error"] = abs_errors_log_model[
    largest_error_indices_log
]

print(
    worst_log_predictions[
        [
            "City",
            "Locality_Type",
            "Super_Area_SqFt",
            "Actual_Price",
            "Predicted_Price",
            "Absolute_Error"
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

plot_data = house_data.sample(
    n=5000,
    random_state=42
)

for locality in plot_data["Locality_Type"].unique():

    subset = plot_data[
        plot_data["Locality_Type"] == locality
    ]

    plt.scatter(
        subset["Super_Area_SqFt"],
        subset["Price_INR_Lakhs"],
        label=locality,
        alpha=0.4
    )

plt.xlabel("Super Area (SqFt)")
plt.ylabel("Price (Lakhs)")
plt.title("Super Area vs Price by Locality Type")
plt.legend()
plt.show()

In [ ]:
area_index = processed_feature_names.index("Super_Area_SqFt")

locality_indices = [
    i for i, name in enumerate(processed_feature_names)
    if name.startswith("Locality_Type_")
]

print("Area index:", area_index)
print("Locality indices:", locality_indices)

print([
    processed_feature_names[i]
    for i in locality_indices
])

In [ ]:
print(encoder.drop)
print(encoder.get_feature_names_out(categorical_features))
print(X_train_processed.shape)

In [ ]:
area_index = processed_feature_names.index("Super_Area_SqFt")

mid_range_index = processed_feature_names.index(
    "Locality_Type_Mid-Range"
)

premium_index = processed_feature_names.index(
    "Locality_Type_Premium"
)

# Create interaction features
train_area_mid = (
    X_train_processed[:, area_index]
    * X_train_processed[:, mid_range_index]
)

train_area_premium = (
    X_train_processed[:, area_index]
    * X_train_processed[:, premium_index]
)

test_area_mid = (
    X_test_processed[:, area_index]
    * X_test_processed[:, mid_range_index]
)

test_area_premium = (
    X_test_processed[:, area_index]
    * X_test_processed[:, premium_index]
)

In [ ]:
print(train_area_mid[:10])
print(train_area_premium[:10])

In [ ]:
X_train_interaction = np.column_stack([
    X_train_processed,
    train_area_mid,
    train_area_premium
])

X_test_interaction = np.column_stack([
    X_test_processed,
    test_area_mid,
    test_area_premium
])

print(X_train_interaction.shape)
print(X_test_interaction.shape)

In [ ]:
X_train_interaction_bias = np.c_[
    np.ones(X_train_interaction.shape[0]),
    X_train_interaction
]

X_test_interaction_bias = np.c_[
    np.ones(X_test_interaction.shape[0]),
    X_test_interaction
]

print(X_train_interaction_bias.shape)
print(X_test_interaction_bias.shape)

In [ ]:
model_interaction = LinearRegression()

model_interaction.fit(
    X_train_interaction_bias,
    y_train.to_numpy()
)

In [ ]:
y_pred_interaction = model_interaction.predict(
    X_test_interaction_bias
)

print(y_pred_interaction[:10])

In [ ]:
errors_interaction = (
    y_test.to_numpy() - y_pred_interaction
)

mae_interaction = np.mean(
    np.abs(errors_interaction)
)

mse_interaction = np.mean(
    errors_interaction ** 2
)

rmse_interaction = np.sqrt(mse_interaction)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_interaction) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_interaction = 1 - (ss_res / ss_tot)

print("MAE :", mae_interaction)
print("RMSE:", rmse_interaction)
print("R²  :", r2_interaction)

In [ ]:
print("Area × Mid-Range coefficient:",
      model_interaction.theta[-2])

print("Area × Premium coefficient:",
      model_interaction.theta[-1])

In [ ]:
area_index = processed_feature_names.index("Super_Area_SqFt")

train_area_squared = (
    X_train_processed[:, area_index] ** 2
)

test_area_squared = (
    X_test_processed[:, area_index] ** 2
)

print(train_area_squared[:10])
print(test_area_squared[:10])

In [ ]:
X_train_polynomial = np.column_stack([
    X_train_processed,
    train_area_squared
])

X_test_polynomial = np.column_stack([
    X_test_processed,
    test_area_squared
])

print(X_train_polynomial.shape)
print(X_test_polynomial.shape)

In [ ]:
X_train_polynomial_bias = np.c_[
    np.ones(X_train_polynomial.shape[0]),
    X_train_polynomial
]

X_test_polynomial_bias = np.c_[
    np.ones(X_test_polynomial.shape[0]),
    X_test_polynomial
]

print(X_train_polynomial_bias.shape)
print(X_test_polynomial_bias.shape)

In [ ]:
model_polynomial = LinearRegression()

model_polynomial.fit(
    X_train_polynomial_bias,
    y_train.to_numpy()
)

In [ ]:
y_pred_polynomial = model_polynomial.predict(
    X_test_polynomial_bias
)

print(y_pred_polynomial[:10])

In [ ]:
errors_polynomial = (
    y_test.to_numpy() - y_pred_polynomial
)

mae_polynomial = np.mean(
    np.abs(errors_polynomial)
)

mse_polynomial = np.mean(
    errors_polynomial ** 2
)

rmse_polynomial = np.sqrt(mse_polynomial)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_polynomial) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_polynomial = 1 - (ss_res / ss_tot)

print("MAE :", mae_polynomial)
print("RMSE:", rmse_polynomial)
print("R²  :", r2_polynomial)

In [ ]:
print("Area coefficient:",
      model_polynomial.theta[1 + area_index])

print("Area² coefficient:",
      model_polynomial.theta[-1])

In [ ]:
error_analysis = X_test.copy()
error_analysis["Absolute_Error"] = np.abs(
    y_test.to_numpy() - y_pred_log_original
)

In [ ]:
error_analysis["Actual_Price"] = y_test.to_numpy()
error_analysis["Predicted_Price"] = y_pred_log_original

In [ ]:
error_analysis.groupby("Locality_Type")["Absolute_Error"].agg(["mean", "count"])

In [ ]:
error_analysis["Relative_Error"] = (
    error_analysis["Absolute_Error"]
    / error_analysis["Actual_Price"]
)

relative_error_by_locality = (
    error_analysis
    .groupby("Locality_Type")["Relative_Error"]
    .agg(["mean", "count"])
)

relative_error_by_locality["mean"] *= 100

print(relative_error_by_locality)

In [ ]:
import matplotlib.pyplot as plt

plot_data_city = house_data.sample(
    n=5000,
    random_state=42
)

for city in plot_data_city["City"].unique():

    subset = plot_data_city[
        plot_data_city["City"] == city
    ]

    plt.scatter(
        subset["Super_Area_SqFt"],
        subset["Price_INR_Lakhs"],
        label=city,
        alpha=0.35
    )

plt.xlabel("Super Area (SqFt)")
plt.ylabel("Price (Lakhs)")
plt.title("Super Area vs Price by City")
plt.legend()
plt.show()

In [ ]:
residuals_log = (
    y_test.to_numpy() - y_pred_log_original
)

plt.scatter(
    X_test["Super_Area_SqFt"],
    residuals_log,
    alpha=0.3
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Super Area (SqFt)")
plt.ylabel("Residual (Lakhs)")
plt.title("Residuals vs Super Area")
plt.show()

In [ ]:
y_test_log = np.log1p(y_test.to_numpy())

residuals_log_space = (
    y_test_log - y_pred_log
)

plt.scatter(
    X_test["Super_Area_SqFt"],
    residuals_log_space,
    alpha=0.3
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Super Area (SqFt)")
plt.ylabel("Log Residual")
plt.title("Log Residuals vs Super Area")
plt.show()


In [ ]:
area_index = processed_feature_names.index("Super_Area_SqFt")
train_area_squared = (
    X_train_processed[:, area_index] ** 2
)

test_area_squared = (
    X_test_processed[:, area_index] ** 2
)

X_train_log_poly = np.column_stack([
    X_train_processed,
    train_area_squared
])

X_test_log_poly = np.column_stack([
    X_test_processed,
    test_area_squared
])

print(X_train_log_poly.shape)
print(X_test_log_poly.shape)

In [ ]:
X_train_log_poly_bias = np.c_[
    np.ones(X_train_log_poly.shape[0]),
    X_train_log_poly
]

X_test_log_poly_bias = np.c_[
    np.ones(X_test_log_poly.shape[0]),
    X_test_log_poly
]

print(X_train_log_poly_bias.shape)
print(X_test_log_poly_bias.shape)

In [ ]:
model_log_poly = LinearRegression()

model_log_poly.fit(
    X_train_log_poly_bias,
    y_train_log
)

In [ ]:
y_pred_log_poly = model_log_poly.predict(
    X_test_log_poly_bias
)

print(y_pred_log_poly[:10])

In [ ]:
y_pred_log_poly_original = np.expm1(
    y_pred_log_poly
)

print(y_pred_log_poly_original[:10])

In [ ]:
errors_log_poly = (
    y_test.to_numpy() - y_pred_log_poly_original
)

mae_log_poly = np.mean(
    np.abs(errors_log_poly)
)

mse_log_poly = np.mean(
    errors_log_poly ** 2
)

rmse_log_poly = np.sqrt(mse_log_poly)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_log_poly_original) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_log_poly = 1 - (ss_res / ss_tot)

print("MAE :", mae_log_poly)
print("RMSE:", rmse_log_poly)
print("R²  :", r2_log_poly)

In [ ]:
y_test_log = np.log1p(y_test.to_numpy())

residuals_log_space = (
    y_test_log - y_pred_log_poly
)

plt.scatter(
    X_test["Super_Area_SqFt"],
    residuals_log_space,
    alpha=0.3
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Super Area (SqFt)")
plt.ylabel("Log Residual")
plt.title("Log Residuals vs Super Area")
plt.show()


In [ ]:
residuals_log_poly = (
    y_test_log - y_pred_log_poly
)

largest_positive_indices = np.argsort(
    residuals_log_poly
)[-10:][::-1]

print(
    residuals_log_poly[largest_positive_indices]
)

In [ ]:
worst_underpredictions = X_test.iloc[
    largest_positive_indices
].copy()

worst_underpredictions["Actual_Price"] = y_test.iloc[
    largest_positive_indices
].values

worst_underpredictions["Predicted_Price"] = (
    y_pred_log_poly_original[
        largest_positive_indices
    ]
)

worst_underpredictions["Log_Residual"] = (
    residuals_log_poly[
        largest_positive_indices
    ]
)

columns_to_inspect = [
    "City",
    "Locality_Type",
    "Property_Type",
    "Furnishing_Status",
    "BHK",
    "Bathrooms",
    "Super_Area_SqFt",
    "Carpet_Area_SqFt",
    "Age_of_Property",
    "Distance_to_Metro_km",
    "Distance_to_City_Center_km",
    "Actual_Price",
    "Predicted_Price",
    "Log_Residual"
]

print(
    worst_underpredictions[
        columns_to_inspect
    ]
)

In [ ]:
worst_underpredictions["Price_per_SqFt"] = (
    worst_underpredictions["Actual_Price"]
    / worst_underpredictions["Super_Area_SqFt"]
)

print(
    worst_underpredictions[
        [
            "City",
            "Locality_Type",
            "Super_Area_SqFt",
            "Actual_Price",
            "Price_per_SqFt",
            "Predicted_Price",
            "Log_Residual"
        ]
    ]
)

In [ ]:
house_data["Price_per_SqFt"] = (
    house_data["Price_INR_Lakhs"]
    / house_data["Super_Area_SqFt"]
)

city_locality_price = (
    house_data
    .groupby(["City", "Locality_Type"])["Price_per_SqFt"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

print(city_locality_price)

In [ ]:
house_data.drop(
    columns=["Price_per_SqFt"],
    inplace=True
)

In [ ]:
city_indices = [
    i for i, name in enumerate(processed_feature_names)
    if name.startswith("City_")
]

locality_indices = [
    i for i, name in enumerate(processed_feature_names)
    if name.startswith("Locality_Type_")
]

print("City indices:", city_indices)
print("City features:", [
    processed_feature_names[i]
    for i in city_indices
])

print("Locality indices:", locality_indices)
print("Locality features:", [
    processed_feature_names[i]
    for i in locality_indices
])

In [ ]:
train_city_locality = []

test_city_locality = []

for city_index in city_indices:

    for locality_index in locality_indices:

        train_interaction = (
            X_train_processed[:, city_index]
            * X_train_processed[:, locality_index]
        )

        test_interaction = (
            X_test_processed[:, city_index]
            * X_test_processed[:, locality_index]
        )

        train_city_locality.append(train_interaction)
        test_city_locality.append(test_interaction)

train_city_locality = np.column_stack(
    train_city_locality
)

test_city_locality = np.column_stack(
    test_city_locality
)

print(train_city_locality.shape)
print(test_city_locality.shape)

In [ ]:
print("City count:", len(city_indices))
print("Locality count:", len(locality_indices))

print("City indices:", city_indices)
print("Locality indices:", locality_indices)

print("City features:")
print([
    processed_feature_names[i]
    for i in city_indices
])

print("Locality features:")
print([
    processed_feature_names[i]
    for i in locality_indices
])

In [ ]:
print(X_train_processed.shape)
print(len(processed_feature_names))

In [ ]:
encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

processed_feature_names = (
    numerical_features
    + list(encoded_feature_names)
)

print(len(processed_feature_names))
print(processed_feature_names)

In [ ]:
city_indices = [
    i
    for i, name in enumerate(processed_feature_names)
    if name.startswith("City_")
]

locality_indices = [
    i
    for i, name in enumerate(processed_feature_names)
    if name.startswith("Locality_Type_")
]

print("City count:", len(city_indices))
print("Locality count:", len(locality_indices))
print("City indices:", city_indices)
print("Locality indices:", locality_indices)

In [ ]:
train_city_locality = []

test_city_locality = []

for city_index in city_indices:

    for locality_index in locality_indices:

        train_interaction = (
            X_train_processed[:, city_index]
            * X_train_processed[:, locality_index]
        )

        test_interaction = (
            X_test_processed[:, city_index]
            * X_test_processed[:, locality_index]
        )

        train_city_locality.append(train_interaction)
        test_city_locality.append(test_interaction)

train_city_locality = np.column_stack(
    train_city_locality
)

test_city_locality = np.column_stack(
    test_city_locality
)

print(train_city_locality.shape)
print(test_city_locality.shape)

In [ ]:
X_train_city_locality = np.column_stack([
    X_train_log_poly,
    train_city_locality
])

X_test_city_locality = np.column_stack([
    X_test_log_poly,
    test_city_locality
])

print(X_train_city_locality.shape)
print(X_test_city_locality.shape)

In [ ]:
X_train_city_locality_bias = np.c_[
    np.ones(X_train_city_locality.shape[0]),
    X_train_city_locality
]

X_test_city_locality_bias = np.c_[
    np.ones(X_test_city_locality.shape[0]),
    X_test_city_locality
]

print(X_train_city_locality_bias.shape)
print(X_test_city_locality_bias.shape)

In [ ]:
model_log_city_locality = LinearRegression()
model_log_city_locality.fit(
    X_train_city_locality_bias,
    y_train_log
)

In [ ]:
y_pred_log_city_locality = (
    model_log_city_locality.predict(
        X_test_city_locality_bias
    )
)

print(y_pred_log_city_locality[:10])

In [ ]:
y_pred_city_locality_original = np.expm1(
    y_pred_log_city_locality
)

print(y_pred_city_locality_original[:10])

In [ ]:
errors_city_locality = (
    y_test.to_numpy()
    - y_pred_city_locality_original
)

mae_city_locality = np.mean(
    np.abs(errors_city_locality)
)

mse_city_locality = np.mean(
    errors_city_locality ** 2
)

rmse_city_locality = np.sqrt(
    mse_city_locality
)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_city_locality_original) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_city_locality = 1 - (
    ss_res / ss_tot
)

print("MAE :", mae_city_locality)
print("RMSE:", rmse_city_locality)
print("R²  :", r2_city_locality)

In [ ]:
current_feature_names = [
    feature
    for feature in processed_feature_names
]

print(len(current_feature_names))
print(current_feature_names)

In [ ]:
def remove_feature(X_train, X_test, feature_names, feature_to_remove):

    index = feature_names.index(feature_to_remove)

    X_train_new = np.delete(
        X_train,
        index,
        axis=1
    )

    X_test_new = np.delete(
        X_test,
        index,
        axis=1
    )

    feature_names_new = [
        feature
        for i, feature in enumerate(feature_names)
        if i != index
    ]

    return X_train_new, X_test_new, feature_names_new

In [ ]:
X_train_log_poly_no_carpet, X_test_log_poly_no_carpet, current_feature_names = remove_feature(
    X_train_log_poly,
    X_test_log_poly,
    current_feature_names,
    "Carpet_Area_SqFt"
)

In [ ]:
X_train_no_carpet_bias = np.c_[
    np.ones(X_train_log_poly_no_carpet.shape[0]),
    X_train_log_poly_no_carpet
]

X_test_no_carpet_bias = np.c_[
    np.ones(X_test_log_poly_no_carpet.shape[0]),
    X_test_log_poly_no_carpet
]

In [ ]:
model_log_poly_no_carpet = LinearRegression()

model_log_poly_no_carpet.fit(
    X_train_no_carpet_bias,
    y_train_log
)

In [ ]:
y_pred_log_no_carpet = model_log_poly_no_carpet.predict(
    X_test_no_carpet_bias
)

In [ ]:
y_pred_no_carpet_original = np.expm1(
    y_pred_log_no_carpet
)

In [ ]:
errors_no_carpet = (
    y_test.to_numpy()
    - y_pred_no_carpet_original
)

mae_no_carpet = np.mean(
    np.abs(errors_no_carpet)
)

mse_no_carpet = np.mean(
    errors_no_carpet ** 2
)

rmse_no_carpet = np.sqrt(
    mse_no_carpet
)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_no_carpet_original) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_no_carpet = 1 - (
    ss_res / ss_tot
)

print("MAE :", mae_no_carpet)
print("RMSE:", rmse_no_carpet)
print("R²  :", r2_no_carpet)

In [ ]:
plt.Figure(figsize=(8, 5))

plt.scatter(
    X_train["Age_of_Property"],
    y_train_log,
    alpha=0.1
)

plt.xlabel("Age of Property (Years)")
plt.ylabel("Log of Price (Lakhs)")
plt.title("Age of Property vs Log of Price")
plt.show()

In [ ]:
age_index = processed_feature_names.index("Age_of_Property")

print(age_index)

In [ ]:
train_age_squared = (
    X_train_processed[:, age_index] ** 2
)

test_age_squared = (
    X_test_processed[:, age_index] ** 2
)

In [ ]:
X_train_age_poly = np.column_stack([
    X_train_log_poly_no_carpet,
    train_age_squared
])

X_test_age_poly = np.column_stack([
    X_test_log_poly_no_carpet,
    test_age_squared
])

print(X_train_age_poly.shape)
print(X_test_age_poly.shape)

In [ ]:
X_train_age_poly_bias = np.c_[
    np.ones(X_train_age_poly.shape[0]),
    X_train_age_poly
]

X_test_age_poly_bias = np.c_[
    np.ones(X_test_age_poly.shape[0]),
    X_test_age_poly
]

In [ ]:
model_log_age_poly = LinearRegression()

model_log_age_poly.fit(
    X_train_age_poly_bias,
    y_train_log
)

In [ ]:
y_pred_log_age_poly = model_log_age_poly.predict(
    X_test_age_poly_bias
)

In [ ]:
y_pred_age_poly_original = np.expm1(
    y_pred_log_age_poly
)

In [ ]:
errors_age_poly = (
    y_test.to_numpy()
    - y_pred_age_poly_original
)

mae_age_poly = np.mean(
    np.abs(errors_age_poly)
)

mse_age_poly = np.mean(
    errors_age_poly ** 2
)

rmse_age_poly = np.sqrt(
    mse_age_poly
)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_age_poly_original) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_age_poly = 1 - (
    ss_res / ss_tot
)

print("MAE :", mae_age_poly)
print("RMSE:", rmse_age_poly)
print("R²  :", r2_age_poly)

In [ ]:
X_train_no_bhk, X_test_no_bhk, current_feature_names = remove_feature(
    X_train_log_poly_no_carpet,
    X_test_log_poly_no_carpet,
    current_feature_names,
    "BHK"
)

In [ ]:
print(X_train_no_bhk.shape)
print(X_test_no_bhk.shape)

In [ ]:
X_train_no_bhk_bias = np.c_[
    np.ones(X_train_no_bhk.shape[0]),
    X_train_no_bhk
]

X_test_no_bhk_bias = np.c_[
    np.ones(X_test_no_bhk.shape[0]),
    X_test_no_bhk
]

print(X_train_no_bhk_bias.shape)
print(X_test_no_bhk_bias.shape)

In [ ]:
model_no_bhk = LinearRegression()
model_no_bhk.fit(
    X_train_no_bhk_bias,
    y_train_log
)

In [ ]:
y_pred_log_no_bhk = model_no_bhk.predict(
    X_test_no_bhk_bias
)

In [ ]:
y_pred_no_bhk_original = np.expm1(
    y_pred_log_no_bhk
)

In [ ]:
error_no_bhk = ( y_test.to_numpy() - y_pred_no_bhk_original)

mae_no_bhk = np.mean(np.abs(error_no_bhk))
rmse_no_bhk = np.sqrt(np.mean(error_no_bhk ** 2))
mse_no_bhk = np.mean(error_no_bhk ** 2)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_bhk_original) ** 2
)

ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2_no_bhk = 1 - (
    ss_res / ss_tot
)

print("MAE :", mae_no_bhk)
print("RMSE:", rmse_no_bhk)
print("R²  :", r2_no_bhk)

In [ ]:
X_train_no_bath, X_test_no_bath, current_feature_names = remove_feature(
    X_train_no_bhk,
    X_test_no_bhk,
    current_feature_names,
    "Bathrooms"
)

In [ ]:
print(X_train_no_bath.shape)
print(X_test_no_bath.shape)

In [ ]:
X_train_no_bath_bias = np.c_[
    np.ones(X_train_no_bath.shape[0]),
    X_train_no_bath
]

X_test_no_bath_bias = np.c_[
    np.ones(X_test_no_bath.shape[0]),
    X_test_no_bath
]

print(X_train_no_bath_bias.shape)
print(X_test_no_bath_bias.shape)

In [ ]:
model_no_bath = LinearRegression()
model_no_bath.fit(X_train_no_bath_bias, y_train_log)

In [ ]:
y_pred_log_no_bath = model_no_bath.predict(
    X_test_no_bath_bias
)

In [ ]:
y_pred_log_no_bath = model_no_bath.predict(X_test_no_bath_bias)

In [ ]:
y_pred_log_no_bath_org = np.expm1(y_pred_log_no_bath)

print(y_pred_log_no_bath[:10])

In [ ]:
errors_no_bath = y_test.to_numpy() - y_pred_log_no_bath_org

mae_no_bath = np.mean(np.abs(errors_no_bath))
mse_no_bath = np.mean(errors_no_bath ** 2)
rmse_no_bath = np.sqrt(mse_no_bath)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_log_no_bath_org) ** 2
)

ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2 = 1 - (ss_res / ss_tot)

print("MAE :", mae_no_bath)
print("RMSE:", rmse_no_bath)
print("R²  :", r2)

In [ ]:
X_train_no_floor, X_test_no_floor, current_feature_names = remove_feature(
    X_train_no_bath,
    X_test_no_bath,
    current_feature_names,
    "Floor_Number"
)

In [ ]:
X_train_no_floor_bias = np.c_[
    np.ones(X_train_no_bath.shape[0]),
    X_train_no_floor
]

X_test_no_floor_bias = np.c_[
    np.ones(X_test_no_floor.shape[0]),
    X_test_no_floor
]

print(X_train_no_floor_bias.shape)
print(X_test_no_floor_bias.shape)

In [ ]:
model_no_floor = LinearRegression()
model_no_floor.fit(
    X_train_no_floor_bias, y_train_log 
)

In [ ]:
y_pred_no_floor = model_no_floor.predict(X_test_no_floor_bias)

In [ ]:
y_pred_no_floor_org = np.expm1(y_pred_no_floor)

In [ ]:
error_no_floor = y_test.to_numpy() - y_pred_no_floor_org

mae_no_floor = np.mean(np.abs(error_no_floor))
mse_no_floor = np.mean(error_no_floor ** 2)
rmse_no_floor = np.sqrt(mse)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_floor_org) ** 2
)

ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2_no_floor = 1 - (ss_res/ss_tot)

print("MAE :", mae_no_floor)
print("RMSE:", rmse_no_bath)
print("R²  :", r2_no_floor)

In [ ]:
X_train_no_total_floor, X_test_no_total_floor, current_feature_names = remove_feature(
    X_train_no_floor,
    X_test_no_floor,
    current_feature_names,
    "Total_Floors"
)

In [ ]:
X_train_no_total_floor_bias = np.c_[
    np.ones(X_train_no_total_floor.shape[0]),
    X_train_no_total_floor
]

X_test_no_total_floor_bias = np.c_[
    np.ones(X_test_no_total_floor.shape[0]),
    X_test_no_total_floor
]

print(X_train_no_total_floor_bias.shape)
print(X_test_no_total_floor_bias.shape)


In [ ]:
model_no_total_floor = LinearRegression()
model_no_total_floor.fit(
    X_train_no_total_floor_bias, y_train_log
)

In [ ]:
y_pred_no_total_floor = model_no_total_floor.predict(X_test_no_total_floor_bias)

In [ ]:
y_pred_no_total_floor_org = np.expm1(y_pred_no_total_floor)

In [ ]:
error_no_total_floor = y_test.to_numpy() - y_pred_no_total_floor_org
mae_no_total_floor = np.mean(np.abs(error_no_total_floor))
mse_no_total_floor = np.mean(error_no_total_floor ** 2)
rmse_no_total_floor = np.sqrt(mse_no_total_floor)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_total_floor_org) ** 2
)

ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2_no_total_floor = 1 - (ss_res/ss_tot)

print("MAE :", mae_no_total_floor)
print("RMSE:", rmse_no_total_floor)
print("R²  :", r2_no_total_floor)

In [ ]:
print("current_feature_names:", current_feature_names)
print("Number of features:", len(current_feature_names))

In [ ]:
X_train_no_parking, X_test_no_parking, current_feature_names = remove_feature(
    X_train_no_total_floor,
    X_test_no_total_floor,
    current_feature_names,
    "Parking"
)

print(X_train_no_parking.shape)
print(X_test_no_parking.shape)
print("Parking" in current_feature_names)

In [ ]:
X_train_no_parking_bias = np.c_[
    np.ones(X_train_no_parking.shape[0]),
    X_train_no_parking
]

X_test_no_parking_bias = np.c_[
    np.ones(X_test_no_parking.shape[0]),
    X_test_no_parking
]

print(X_train_no_parking_bias.shape)
print(X_test_no_parking_bias.shape)

In [ ]:
model_no_parking = LinearRegression()
model_no_parking.fit(
    X_train_no_parking_bias, y_train_log
)

In [ ]:
y_pred_no_parking = model_no_parking.predict(X_test_no_parking_bias)

In [ ]:
y_pred_no_parking_org = np.expm1(y_pred_no_parking)

In [ ]:
error_no_parking = y_test.to_numpy() - np.expm1(y_pred_no_parking)
mae_no_parking = np.mean(np.abs(error_no_parking))
mse_no_parking = np.mean(error_no_parking ** 2)
rmse_no_parking = np.sqrt(mse_no_parking)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_parking_org) ** 2
)
ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)
r2_no_parking = 1 - (ss_res/ss_tot)

print("MAE :", mae_no_parking)
print("RMSE:", rmse_no_parking)
print("R²  :", r2_no_parking)

In [ ]:
X_train_no_lift, X_test_no_lift, current_feature_names = remove_feature(
    X_train_no_parking,
    X_test_no_parking,
    current_feature_names,
    "Lift_Available"
)

print(X_train_no_lift.shape)
print(X_test_no_lift.shape)
print("Lift_Available" in current_feature_names)

In [ ]:
X_train_no_lift_bias = np.c_[
    np.ones(X_train_no_lift.shape[0]),
    X_train_no_lift
]
X_test_no_lift_bias = np.c_[
    np.ones(X_test_no_lift.shape[0]),
    X_test_no_lift
]

print(X_train_no_lift_bias.shape)
print(X_test_no_lift_bias.shape)

In [ ]:
model_no_lift = LinearRegression()
model_no_lift.fit(
    X_train_no_lift_bias, y_train_log
)

In [ ]:
y_pred_no_lift = model_no_lift.predict(X_test_no_lift_bias)

In [ ]:
y_pred_no_lift_org = np.expm1(y_pred_no_lift)

In [ ]:
error_no_lift = y_test.to_numpy() - y_pred_no_lift_org
mae_no_lift = np.mean(np.abs(error_no_lift))
mse_no_lift = np.mean(error_no_lift ** 2)
rmse_no_lift = np.sqrt(mse_no_lift)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_lift_org) ** 2
)
ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2_no_lift = 1 - (ss_res/ss_tot)

print("MAE :", mae_no_lift)
print("RMSE:", rmse_no_lift)
print("R²  :", r2_no_lift)

In [ ]:
X_train_no_gated, X_test_no_gated, current_feature_names = remove_feature(
    X_train_no_lift,
    X_test_no_lift,
    current_feature_names,
    "Gated_Community"
)

print(X_train_no_gated.shape)
print(X_test_no_gated.shape)
print("Gated_Community" in current_feature_names)

In [ ]:
X_train_no_gated_bias = np.c_[
    np.ones(X_train_no_gated.shape[0]),
    X_train_no_gated
]
X_test_no_gated_bias = np.c_[
    np.ones(X_test_no_gated.shape[0]),
    X_test_no_gated
]   

print(X_train_no_gated_bias.shape)
print(X_test_no_gated_bias.shape)

In [ ]:
model_no_gated = LinearRegression()
model_no_gated.fit(
    X_train_no_gated_bias, y_train_log
)

In [ ]:
y_pred_no_gated = model_no_gated.predict(X_test_no_gated_bias)

In [ ]:
y_pred_no_gated_org = np.expm1(y_pred_no_gated)

In [ ]:
error_no_gated = y_test.to_numpy() - y_pred_no_gated_org
mae_no_gated = np.mean(np.abs(error_no_gated))
mse_no_gated = np.mean(error_no_gated ** 2)
rmse_no_gated = np.sqrt(mse_no_gated)
ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_gated_org) ** 2
)
ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)
r2_no_gated = 1 - (ss_res/ss_tot)   

print("MAE :", mae_no_gated)
print("RMSE:", rmse_no_gated)
print("R²  :", r2_no_gated)

In [ ]:
X_train_no_metro, X_test_no_metro, current_feature_names = remove_feature(
    X_train_no_gated,
    X_test_no_gated,
    current_feature_names,
    "Distance_to_Metro_km"
)

print(X_train_no_metro.shape)
print(X_test_no_metro.shape)

In [ ]:
X_train_no_metro_bias = np.c_[
    np.ones(X_train_no_metro.shape[0]),
    X_train_no_metro
]  

X_test_no_metro_bias = np.c_[
    np.ones(X_test_no_metro.shape[0]),
    X_test_no_metro
]

print(X_train_no_metro_bias.shape)
print(X_test_no_metro_bias.shape)

In [ ]:
model_no_metro = LinearRegression()
model_no_metro.fit(
    X_train_no_metro_bias, y_train_log
)

In [ ]:
y_pred_no_metro = model_no_metro.predict(X_test_no_metro_bias)

In [ ]:
y_pred_no_metro_org = np.expm1(y_pred_no_metro)

In [ ]:
error_no_metro = y_test.to_numpy() - y_pred_no_metro_org
mae_no_metro = np.mean(np.abs(error_no_metro))
mse_no_metro = np.mean(error_no_metro ** 2)
rmse_no_metro = np.sqrt(mse_no_metro)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_metro_org) ** 2
)
ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)
r2_no_metro = 1 - (ss_res/ss_tot)

print("MAE :", mae_no_metro)
print("RMSE:", rmse_no_metro)
print("R²  :", r2_no_metro)

In [ ]:
X_train_no_city_center, X_test_no_city_center, current_feature_names = remove_feature(
    X_train_no_metro,
    X_test_no_metro,
    current_feature_names,
    "Distance_to_City_Center_km"
)

print(X_train_no_city_center.shape)
print(X_test_no_city_center.shape)
print("Distance_to_City_Center_km" in current_feature_names)

In [ ]:
X_train_no_city_centre_bias = np.c_[
    np.ones(X_train_no_city_center.shape[0]),
    X_train_no_city_center
]
X_test_no_city_centre_bias = np.c_[
    np.ones(X_test_no_city_center.shape[0]),
    X_test_no_city_center
]

print(X_train_no_city_centre_bias.shape)
print(X_test_no_city_centre_bias.shape)

In [ ]:
model_no_city_center = LinearRegression()
model_no_city_center.fit(
    X_train_no_city_centre_bias, y_train_log
)

In [ ]:
y_pred_no_city_center = model_no_city_center.predict(X_test_no_city_centre_bias)

In [ ]:
y_pred_no_city_center_org = np.expm1(y_pred_no_city_center)

In [ ]:
error_no_city_center = y_test.to_numpy() - y_pred_no_city_center_org
mae_no_city_center = np.mean(np.abs(error_no_city_center))
mse_no_city_center = np.mean(error_no_city_center ** 2)
rmse_no_city_center = np.sqrt(mse_no_city_center)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_no_city_center_org) ** 2
)
ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)
r2_no_city_center = 1 - (ss_res/ss_tot)

print("MAE :", mae_no_city_center)
print("RMSE:", rmse_no_city_center)
print("R²  :", r2_no_city_center)  

In [ ]:
keep_features = [
    "Super_Area_SqFt",
    "Age_of_Property",
    "Distance_to_City_Center_km"
]

keep_features += [
    feature
    for feature in processed_feature_names
    if feature.startswith("City_")
    or feature.startswith("Locality_Type_")
    or feature.startswith("Property_Type_")
    or feature.startswith("Furnishing_Status_")
]

print("Number of base features:", len(keep_features))
print(keep_features)

In [ ]:
keep_indices = [
    processed_feature_names.index(feature)
    for feature in keep_features
]

X_train_clean = X_train_processed[:, keep_indices]
X_test_clean = X_test_processed[:, keep_indices]

print(X_train_clean.shape)
print(X_test_clean.shape)

In [ ]:
area_clean_index = keep_features.index("Super_Area_SqFt")

print(area_clean_index)

In [ ]:
train_area_squared_clean = (
    X_train_clean[:, area_clean_index] ** 2
)

test_area_squared_clean = (
    X_test_clean[:, area_clean_index] ** 2
)

In [ ]:
X_train_final = np.column_stack([
    X_train_clean,
    train_area_squared_clean
])

X_test_final = np.column_stack([
    X_test_clean,
    test_area_squared_clean
])

print(X_train_final.shape)
print(X_test_final.shape)

In [ ]:
final_feature_names = keep_features + ["Super_Area_SqFt²"]

print(len(final_feature_names))
print(final_feature_names)

In [ ]:
X_train_final_bias = np.c_[
    np.ones(X_train_final.shape[0]),
    X_train_final
]

X_test_final_bias = np.c_[
    np.ones(X_test_final.shape[0]),
    X_test_final
]

print(X_train_final_bias.shape)
print(X_test_final_bias.shape)

In [ ]:
model_final_base = LinearRegression()

model_final_base.fit(
    X_train_final_bias,
    y_train_log
)

print("Number of parameters:", len(model_final_base.theta))

In [ ]:
y_pred_log_final_base = model_final_base.predict(
    X_test_final_bias
)

y_pred_final_base = np.expm1(
    y_pred_log_final_base
)

In [ ]:
errors_final_base = (
    y_test.to_numpy()
    - y_pred_final_base
)

mae_final_base = np.mean(
    np.abs(errors_final_base)
)

mse_final_base = np.mean(
    errors_final_base ** 2
)

rmse_final_base = np.sqrt(
    mse_final_base
)

y_test_array = y_test.to_numpy()
y_mean = np.mean(y_test_array)

ss_res = np.sum(
    (y_test_array - y_pred_final_base) ** 2
)

ss_tot = np.sum(
    (y_test_array - y_mean) ** 2
)

r2_final_base = 1 - (
    ss_res / ss_tot
)

print("MAE :", mae_final_base)
print("RMSE:", rmse_final_base)
print("R²  :", r2_final_base)

In [ ]:
distance_index = final_feature_names.index(
    "Distance_to_City_Center_km"
)

plt.figure(figsize=(8, 5))

plt.scatter(
    X_train_final[:, distance_index],
    y_train_log,
    alpha=0.1
)

plt.xlabel("Standardized Distance to City Center")
plt.ylabel("log(1 + Price)")
plt.title("Distance to City Center vs Log Price")

plt.show()

In [ ]:
distance_index = final_feature_names.index(
    "Distance_to_City_Center_km"
)

print(distance_index)

In [ ]:
train_distance_squared = (
    X_train_final[:, distance_index] ** 2
)

test_distance_squared = (
    X_test_final[:, distance_index] ** 2
)

In [ ]:
X_train_distance_poly = np.column_stack([
    X_train_final,
    train_distance_squared
])

X_test_distance_poly = np.column_stack([
    X_test_final,
    test_distance_squared
])

print(X_train_distance_poly.shape)
print(X_test_distance_poly.shape)

In [ ]:
X_train_distance_poly_bias = np.c_[
    np.ones(X_train_distance_poly.shape[0]),
    X_train_distance_poly
]
X_test_distance_poly_bias = np.c_[
    np.ones(X_test_distance_poly.shape[0]),
    X_test_distance_poly
]

print(X_train_distance_poly_bias.shape)
print(X_test_distance_poly_bias.shape)

In [ ]:
model_distance_poly = LinearRegression()
model_distance_poly.fit(
    X_train_distance_poly_bias,
    y_train_log
)

In [ ]:
y_pred_distance_poly = model_distance_poly.predict(
    X_test_distance_poly_bias
)

In [ ]:
y_pred_distance_poly_org = np.expm1(y_pred_distance_poly)

In [ ]:
errors_distance_poly = (
    y_test.to_numpy()
    - y_pred_distance_poly_org
)

mae_distance_poly = np.mean(
    np.abs(errors_distance_poly)
)   

mse_distance_poly = np.mean(
    errors_distance_poly ** 2
)

rmse_distance_poly = np.sqrt(
    mse_distance_poly
)

ss_res = np.sum(
    (y_test.to_numpy() - y_pred_distance_poly_org) ** 2
)

ss_tot = np.sum(
    (y_test.to_numpy() - np.mean(y_test.to_numpy())) ** 2
)

r2_distance_poly = 1 - (ss_res/ss_tot)

print("MAE :", mae_distance_poly)
print("RMSE:", rmse_distance_poly)
print("R²  :", r2_distance_poly)

In [ ]:
residuals = (
    y_test.to_numpy()
    - y_pred_final_base
)

print("Mean residual:", np.mean(residuals))
print("Min residual:", np.min(residuals))
print("Max residual:", np.max(residuals))

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    y_pred_final_base,
    residuals,
    alpha=0.1
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Predicted Price (Lakhs)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Predicted Price")

plt.show()

# Decision Tree Regression

In [ ]:
tree_numeric_features = [
    "Super_Area_SqFt",
    "Age_of_Property",
    "Distance_to_City_Center_km"
]

In [ ]:
tree_categorical_features = [
    "City",
    "Locality_Type",
    "Property_Type",
    "Furnishing_Status"
]

In [ ]:
X_train_tree_cat = encoder.transform(
    X_train[tree_categorical_features]
)

X_test_tree_cat = encoder.transform(
    X_test[tree_categorical_features]
)

In [ ]:
X_train_tree_num = X_train[
    tree_numeric_features
].to_numpy()

X_test_tree_num = X_test[
    tree_numeric_features
].to_numpy()

In [ ]:
X_train_tree = np.hstack([
    X_train_tree_num,
    X_train_tree_cat.toarray()
])

X_test_tree = np.hstack([
    X_test_tree_num,
    X_test_tree_cat.toarray()
])

print(X_train_tree.shape)
print(X_test_tree.shape)

In [ ]:
max_depth = None

In [ ]:
class Node:

    def __init__(self,feature_index=None,threshold=None,prediction=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.prediction = prediction
        self.left = None
        self.right = None

In [ ]:
class DecisionTreeRegression:
  
    def fit(self,X,y):
      
        def calculate_sse(y):
            mean_y = np.mean(y)
            sse = np.sum((y-mean_y)**2)
            return sse
      
        def calculate_split_sse(X,y,threshold):
        
            left_mask = X < threshold
            right_mask = X >= threshold

            left_y = y[left_mask]
            right_y = y[right_mask]

            left_sse = calculate_sse(left_y)
            right_sse = calculate_sse(right_y)

            return left_sse + right_sse
      
        def find_best_split(X,y):
            min_sse = np.inf
            best_threshold = None

            for i in range(len(X)-1):
                threshold = (X[i] + X[i+1])/2
                sse = calculate_split_sse(X,y,threshold)
                if sse < min_sse:
                    min_sse = sse
                    best_threshold = threshold
        
            return best_threshold, min_sse

        def split_data(X, y, threshold):

            left_mask = X < threshold
            right_mask = X >= threshold

            X_left = X[left_mask]
            y_left = y[left_mask]

            X_right = X[right_mask]
            y_right = y[right_mask]

            

        def create_leaf(y):
            return np.mean(y)



        def build_tree(X, y):
            if len(y) == 1:
                prediction = create_leaf(y)
                return Node(prediction=prediction)

            best_threshold, min_sse = find_best_split(X, y)

            X_left, y_left, X_right, y_right = split_data(
                    X,
                    y,
                    best_threshold
                )
            
            node = Node(threshold=best_threshold)

            node.left = build_tree(X_left, y_left)

            node.right = build_tree(X_right, y_right)
            
        
            return node
        
        self.root = build_tree(X,y)
        
        
    def predict_one(self, X, node):

        if node.prediction is not None:
            return node.prediction

        if X < node.threshold:
            return self.predict_one(X, node.left)

        else:
            return self.predict_one(X, node.right)



In [1]:
print("Decision Tree Regression model created successfully not successfully.")

Decision Tree Regression model created successfully not successfully.
